# Fine-tuning restrito — RECLin-PT (Colab T4) · frente EXPERIMENTAL

Treina um encoder **só no espaço restrito de candidatos** (pares cujo `e1` é pista de negação segundo o léxico induzido do train, `min_freq=3`, a mesma calibração do filtro da fase 2) e remapeia as predições ao espaço completo para comparar com os 4 baselines do Cap. 6.

**Isolamento.** Todo o código está em `src/finetuning_restrito/` e todo resultado vai para `results/finetuning_restrito/`. Nenhum arquivo existente do repositório é modificado, e o script recusa saída em qualquer outro lugar de `results/`. Os checkpoints vão para pastas **novas** no Drive (`checkpoints_restrito_<encoder>_seed<N>`) e o backup no HF Hub vai para repositórios **novos** (`reclin-pt-restrito-<encoder>-seed<N>`); o script recusa uma pasta de checkpoint de baseline e um repo de backup sem `restrito` no nome.

**Ordem obrigatória.**
1. **Tarefa 2 (sanidade):** 1 encoder × sementes 42 e 43 (seção 7). Depois, o critério de parada (seção 8).
2. **Tarefa 3 (escala):** 2 encoders × 10 sementes (seção 9). A célula **se recusa a rodar** se o critério de parada não tiver passado.
3. **Tarefa 4 (avaliação):** seção 10, só com as 20 execuções prontas.

Custo estimado numa T4: ~107 passos por época (eram 2.386 nos baselines), ~3 min por época, **~30 min por execução de 10 épocas**. Sanidade ≈ 1 h; escala (18 execuções novas) ≈ 9 h, em mais de uma sessão (a retomada é automática).

Antes de rodar: **Ambiente de execução → GPU (T4)** e os secrets `GITHUB_PAT` e (opcional) `HF_TOKEN`.

## 1. Clonar o repositório

In [ ]:
import os, subprocess
from google.colab import userdata

TOKEN = userdata.get('GITHUB_PAT')           # secret do Colab, nunca hard-coded
REPO_NAME = 'RECLin-PT'
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

url = f'https://{TOKEN}@github.com/angeloalsf/{REPO_NAME}'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)
subprocess.run(['git', '-C', REPO, 'config', 'user.name', GIT_USER_NAME], check=True)
subprocess.run(['git', '-C', REPO, 'config', 'user.email', GIT_USER_EMAIL], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', url], check=True)
assert os.path.isfile(os.path.join(REPO, 'src', 'finetuning_restrito', 'train_restrito.py')), \
    'src/finetuning_restrito/ nao esta no clone: faca commit/push da pasta antes.'
print('Repo pronto em', REPO)

## 2. Verificar a GPU

In [ ]:
!nvidia-smi
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'

## 3. Parâmetros da execução

`ENCODER` da sanidade: **BERTimbau** por padrão, por ser o mais rápido (60% dos parâmetros do BioBERTpt; ~1,5 s por passo contra ~1,65 s nos baselines). Os hiperparâmetros são os dos baselines (`max_gap 25`, `max_length 128`, `batch 64`, `lr 2e-5`, `balanced`), exceto `--epochs 10`. Não mude nada entre a sanidade e a escala: a escala tem de usar **os hiperparâmetros validados na sanidade**.

In [ ]:
ENCODER_SANIDADE = 'bertimbau'      # 'bertimbau' ou 'biobertpt'
SEEDS_SANIDADE = [42, 43]
EPOCHS = 10

ENCODERS_ESCALA = ['biobertpt', 'bertimbau']
SEEDS_ESCALA = list(range(42, 52))  # 10 sementes; 42 e 43 do encoder da sanidade sao reaproveitadas

from google.colab import drive
drive.mount('/content/drive')
CKPT_BASE = f'/content/drive/MyDrive/{REPO_NAME}'
def ckpt_dir(enc, seed):
    return f'{CKPT_BASE}/checkpoints_restrito_{enc}_seed{seed}'
print('Checkpoints em', CKPT_BASE + '/checkpoints_restrito_<encoder>_seed<N>')

## 4. Dependências

Mesmo `requirements.txt` dos baselines, com as versões fixadas (`torch==2.11.0`, `transformers==5.16.1` etc.). Se esta célula começar a baixar o `torch`, a imagem do Colab mudou de versão: **pare** e registre, porque a comparação com os baselines passa a ter ambiente diferente (o campo `environment` de cada JSON mostra isso).

In [ ]:
!pip install -q -r $REPO/requirements.txt
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)

## 5. Verificação rápida em CPU (Tarefa 1)

Roda as 7 checagens do espaço restrito e do remapeamento, sem GPU e sem pesos. Precisa passar antes de gastar GPU.

In [ ]:
%cd $REPO
!python src/finetuning_restrito/test_remapeamento.py

## 6. Token do Hugging Face Hub (backup automático, opcional)

Mesmo mecanismo dos baselines (`src/hf_backup.py`, repositório **privado**), mas em repositórios **separados**: `angeloalsf/reclin-pt-restrito-<encoder>-seed<N>`. O token viaja só pelo ambiente, nunca pela linha de comando.

In [ ]:
HF_USER = 'angeloalsf'
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception as erro:
    hf_token = None
    print('Secret HF_TOKEN indisponivel (%s).' % type(erro).__name__)
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN exportado para o ambiente (o valor nunca e impresso).')
else:
    print('BACKUP NO HF HUB DESLIGADO: o treino roda igual, so sem a copia extra.')

def hf_flags(enc, seed):
    return f'--hf-backup-repo {HF_USER}/reclin-pt-restrito-{enc}-seed{seed}' if hf_token else ''

def treinar(enc, seed):
    out = f'results/finetuning_restrito/restrito_{enc}_seed{seed}.json'
    if os.path.isfile(os.path.join(REPO, out)):
        print(f'{out} ja existe: pulando (apague/renomeie para retreinar).')
        return out
    cmd = (f'cd {REPO} && PYTHONHASHSEED=0 python -u src/finetuning_restrito/train_restrito.py '
           f'--encoder {enc} --seed {seed} --epochs {EPOCHS} '
           f'--ckpt-dir "{ckpt_dir(enc, seed)}" {hf_flags(enc, seed)}')
    print(cmd.replace(REPO, '$REPO'))
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f'treino {enc} seed {seed} falhou (codigo {r.returncode})'
    return out

def commit(paths, msg):
    existentes = [p for p in paths if os.path.exists(os.path.join(REPO, p))]
    for p in existentes:
        subprocess.run(['git', '-C', REPO, 'add', '-f', p], check=False)
    subprocess.run(['git', '-C', REPO, 'commit', '-m', msg], check=False)
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)
    subprocess.run(['git', '-C', REPO, 'push'], check=False)

def saidas(enc, seed):
    stem = f'results/finetuning_restrito/restrito_{enc}_seed{seed}'
    return [stem + s for s in ('.json', '.preds.json', '.dev_preds.json',
                               '.test_evals.jsonl', '.train_log.txt')]

## 7. Tarefa 2 — sanidade (1 encoder × sementes 42 e 43)

Se o runtime cair, rode a célula de novo: cada execução retoma do `last_checkpoint` no Drive, e execuções já concluídas são puladas.

In [ ]:
for seed in SEEDS_SANIDADE:
    treinar(ENCODER_SANIDADE, seed)
    commit(saidas(ENCODER_SANIDADE, seed),
           f'finetuning_restrito: sanidade {ENCODER_SANIDADE} seed {seed} (Colab T4)')

## 8. Critério de parada (DEV, contra o BASELINE)

Regra fixada antes de rodar: **passa** se o F1 de `negation_of` no DEV do restrito superar o do baseline do mesmo encoder e da mesma semente, **nas duas sementes**. Não lê o TEST. Se **não passar, pare aqui**: esse é o veredito da frente.

In [ ]:
!cd $REPO && python src/finetuning_restrito/criterio_parada.py --encoder $ENCODER_SANIDADE --seeds 42 43
import json
from IPython.display import Markdown, display
crit = f'{REPO}/results/finetuning_restrito/CRITERIO_PARADA_tarefa2_{ENCODER_SANIDADE}'
display(Markdown(open(crit + '.md', encoding='utf-8').read()))
commit([f'results/finetuning_restrito/CRITERIO_PARADA_tarefa2_{ENCODER_SANIDADE}.json',
        f'results/finetuning_restrito/CRITERIO_PARADA_tarefa2_{ENCODER_SANIDADE}.md',
        'results/finetuning_restrito/significancia_dev'],
       f'finetuning_restrito: criterio de parada da Tarefa 2 ({ENCODER_SANIDADE})')

## 9. Tarefa 3 — escala (2 encoders × 10 sementes) · SÓ SE A SEÇÃO 8 PASSOU

A célula lê o JSON do critério de parada e **se recusa a rodar** se ele não tiver passado. ~9 h de T4 no total; rode em várias sessões (execuções prontas são puladas, as interrompidas retomam).

In [ ]:
crit_json = json.load(open(crit + '.json', encoding='utf-8'))
assert crit_json['passa'], 'Criterio de parada NAO passou: a Tarefa 3 nao deve rodar. ' + crit_json['veredito']
for enc in ENCODERS_ESCALA:
    for seed in SEEDS_ESCALA:
        treinar(enc, seed)
        commit(saidas(enc, seed), f'finetuning_restrito: escala {enc} seed {seed} (Colab T4)')

## 10. Tarefa 4 — avaliação final (CPU, minutos)

`src/significance.py` inalterado, cada execução contra os 4 baselines (comparação principal) e as 4 execuções filtradas da fase 2 (secundária), mais a agregação por semente.

In [ ]:
!cd $REPO && python src/finetuning_restrito/avaliacao_final.py --seeds 42 43 44 45 46 47 48 49 50 51
display(Markdown(open(f'{REPO}/results/finetuning_restrito/AVALIACAO_final.md', encoding='utf-8').read()))
commit(['results/finetuning_restrito/AVALIACAO_final.json',
        'results/finetuning_restrito/AVALIACAO_final.md',
        'results/finetuning_restrito/significancia'],
       'finetuning_restrito: avaliacao final (Tarefa 4)')